# 04 — Baseline 2: Greedy Divert Heuristic

---

### What this notebook does

For each focus order it looks at the other DCs. If one of them can fill the order better, and the
rules allow it, the order is moved there. It handles **one order at a time and never goes back**.
That is what *greedy* means.

Two sort orders are run, because a greedy answer depends on who goes first:

* **2A** biggest revenue first — protect the big money.
* **2B** biggest shortfall first — fix the worst problems.

Only the sort changes. The checks and the objective are identical.

### What it does not do

No cleaning and no rule definitions — those come from `dom_model`. The only thing added here is
the search itself.

## 1. Setup

In [1]:
import os, sys, time, json
import numpy as np
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)

In [2]:
import os, glob, zipfile

# Colab keeps nothing between runtimes, so the shared folder goes on Drive when we can.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/DOM"
except Exception:
    WORK = os.path.abspath("dom_work")

CLEAN   = f"{WORK}/clean"      # notebook 02 writes here
RESULTS = f"{WORK}/results"    # notebooks 03, 04, 05 write here
os.makedirs(CLEAN, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)
print("work folder:", WORK)

Mounted at /content/drive
work folder: /content/drive/MyDrive/DOM


In [3]:
# The model lives in exactly one file, written by 02_data_cleaning.
# Data, constraints C1-C7 and the objective all arrive from here.
sys.path.insert(0, WORK)
os.environ["DOM_CLEAN"] = CLEAN
from dom_model import *

print("orders:", len(HEAD), "| focus:", len(FOCUS), "| clean:", len(CLEAN_ORDERS))
print("DCs:", DCS, "| days:", NT)
print("CFG:", CFG)

orders: 1109 | focus: 472 | clean: 637
DCs: [5083, 5385, 5410, 5420, 5490, 5620, 5641, 5773] | days: 32
CFG: {'MIN_FILL_LIFT_PP': 0.05, 'MIN_CASE_LIFT': 100.0, 'FORWARD_COVER_DAYS': 5, 'LEAD_TIME_MILES': 500.0, 'DOCKS_PER_ORDER': 1, 'SAFETY_STOCK_FRAC': 0.0, 'REQUIRE_OBJ_GAIN': True}


### 1.1 The starting point

The same `stage_A()` that `03` reported as Baseline 1. `POOL_A` is the stock left over once
everyone sits at their own DC — that is what a divert has to draw from.

In [4]:
POOL_A, DEF = stage_A()
BASE1 = {gf: dict(DEF[gf], diverted=False, chosen_dc=DEF[gf]["dc"], lift=0.0) for gf in DEF}
print("focus orders to decide:", len(FOCUS))
print("priority-8 orders in total:", sum(1 for g in HEAD if HEAD[g]["prio"] == 8))
print("priority-8 orders in FOCUS:", sum(1 for g in FOCUS if HEAD[g]["prio"] == 8))

focus orders to decide: 472
priority-8 orders in total: 12
priority-8 orders in FOCUS: 0


## 2. The greedy pass

One pass. For each focus order, in the given sort order:

1. look at every other DC,
2. check in turn — does it stock the SKUs → is the new date OK → is there stock (**C3**) → does it
   pass the gate (**C4**) → is there a dock (**C6**) → is there pick room (**C5**),
3. keep the DC with the best objective,
4. if that beats staying put, give the stock back at the old DC and take it at the new one.

Every rejection goes into a log, so a planner can see why an order was not moved. All seven checks
are the functions imported from `dom_model` — none of them is redefined here.

In [5]:
def greedy(order_key, cfg=None):
    cfg = {**CFG, **(cfg or {})}                      # allow one-off setting changes
    P   = {k: v.copy() for k, v in POOL_A.items()}    # stock left after stage A
    dk, cp_r, pp_r = DOCK0.copy(), CP0.copy(), PP0.copy()   # free docks and picks

    for gf, r in DEF.items():                         # every default order already
        i, j = DCIX[r["dc"]], r["t"]                  # holds one dock slot
        if DOCK_HAS[i, j]:
            dk[i, j] = max(0.0, dk[i, j] - cfg["DOCKS_PER_ORDER"])

    res = {gf: dict(DEF[gf], diverted=False, chosen_dc=DEF[gf]["dc"], lift=0.0)
           for gf in DEF}                             # start from the default answer
    log = []                                          # why each try failed

    for gf in sorted(FOCUS, key=order_key):           # <-- the greedy order
        h, base = HEAD[gf], DEF[gf]
        d0, t0_ = base["dc"], base["t"]               # its current DC and day
        best = None                                   # best alternative so far

        for d in DCS:                                 # try every other DC
            if d == d0:
                continue                              # skip its own DC
            if not all(s in SKU_AT_DC[d] for s, _, _, _ in LINES[gf]):
                log.append((gf, d, "sku_not_carried")); continue   # DC lacks a SKU
            p, err = revised_pgi(gf, d)               # new ship day
            if err:
                log.append((gf, d, err)); continue
            t = DIX[p]
            fills, by, tot, rev = evaluate(P, gf, d, t,
                                           cfg["FORWARD_COVER_DAYS"],   # C3
                                           cfg["SAFETY_STOCK_FRAC"])
            lift = tot - base["filled"]               # extra cases we would gain
            g = passes_gate(lift, h["ordered_cases"], cfg)             # C4
            if g:
                log.append((gf, d, g)); continue
            i = DCIX[d]
            if DOCK_HAS[i, t] and dk[i, t] < cfg["DOCKS_PER_ORDER"]:   # C6
                log.append((gf, d, "no_dock")); continue
            cp, pp = picks(fills)                                      # C5
            if cp > cp_r[i, t] or pp > pp_r[i, t]:
                log.append((gf, d, "no_pick_capacity")); continue
            cand = dict(dc=d, t=t, fills=fills, by=by, filled=tot, revenue=rev,
                        pen=penalty_of(gf, by, tot),                   # C7
                        ship=SHIP.get((d, h["zipc"]), 0.0),
                        cp=cp, pp=pp, lift=lift,
                        cof=tot / h["ordered_cases"] if h["ordered_cases"] else 0.0)
            if best is None or objective(cand) > objective(best):
                best = cand                            # keep the richest option

        if best is None:                               # nothing was feasible
            log.append((gf, None, "no_feasible_alternate")); continue
        if cfg["REQUIRE_OBJ_GAIN"] and objective(best) <= objective(base):
            log.append((gf, best["dc"], "no_objective_gain")); continue  # not worth it

        for s, q, _, _ in base["fills"]:
            give(P, d0, s, t0_, q)                     # hand the stock back to the old DC
        if DOCK_HAS[DCIX[d0], t0_]:
            dk[DCIX[d0], t0_] += cfg["DOCKS_PER_ORDER"]    # and its dock slot
        for s, q, _, _ in best["fills"]:
            take(P, best["dc"], s, best["t"], q)       # take stock at the new DC
        i = DCIX[best["dc"]]
        if DOCK_HAS[i, best["t"]]:
            dk[i, best["t"]] -= cfg["DOCKS_PER_ORDER"] # use a dock there
        cp_r[i, best["t"]] -= best["cp"]               # use pick capacity there
        pp_r[i, best["t"]] -= best["pp"]

        res[gf] = dict(best, diverted=True, chosen_dc=best["dc"])   # record the move

    return res, pd.DataFrame(log, columns=["order","dc","reason"])

## 3. Run both sort orders

In [6]:
KEYS = {
    "2A · by order value"      : lambda g: -HEAD[g]["revenue"],   # minus = biggest first
    "2B · by shortage severity": lambda g: -(HEAD[g]["ordered_cases"] - DEF[g]["filled"]),
}

runs, rows = {}, []
for name, k in KEYS.items():
    t0 = time.time()
    r, lg = greedy(k)                                  # run one greedy pass
    rt = round(time.time() - t0, 2)
    rows.append(metrics(r, f"Greedy {name}", rt))
    runs[name] = (r, lg)                               # save result + reject log
    print(f"{name:28s} moves = {rows[-1]['orders_diverted']:3d}   {rt}s")

2A · by order value          moves =  41   0.4s
2B · by shortage severity    moves =  39   0.42s


### 3.1 Against Baseline 1

The greedy buys fill rate and pays for it in freight. The objective is what decides whether that
was a good trade.

In [7]:
comp = pd.DataFrame([metrics(BASE1, "Baseline 1 · default assignment", 0.0)] + rows) \
         .set_index("scenario")
b1 = comp.iloc[0]
print(pd.DataFrame({
    "Δ objective ($)"  : comp["objective_focus"] - b1["objective_focus"],
    "Δ fill rate (pts)": (comp["fill_rate"] - b1["fill_rate"]) * 100,
    "Δ penalty ($)"    : comp["penalty_cost"]  - b1["penalty_cost"],
    "Δ shipping ($)"   : comp["shipping_cost"] - b1["shipping_cost"],
    "moves"            : comp["orders_diverted"],
}).round(2).to_string())

                                  Δ objective ($)  Δ fill rate (pts)  Δ penalty ($)  Δ shipping ($)  moves
scenario                                                                                                  
Baseline 1 · default assignment              0.00               0.00           0.00             0.0      0
Greedy 2A · by order value              444610.19               0.81       -9490.47         72686.0     41
Greedy 2B · by shortage severity        420484.74               0.79       -9320.11         71337.0     39


### 3.2 Why orders were not moved

This is the diagnostic a planner actually wants: which rule is doing the blocking. Run 2A is used
for the log.

In [8]:
res2A, log2A = runs["2A · by order value"]
rej = log2A.reason.value_counts().rename("count").to_frame()
rej["share"] = (rej["count"] / rej["count"].sum()).map(lambda v: f"{v*100:.1f}%")
print(rej.to_string())

                       count  share
reason                             
fail_5pct               1682  46.0%
sku_not_carried         1457  39.8%
no_feasible_alternate    423  11.6%
fail_100cases             37   1.0%
pgi_out_of_horizon        24   0.7%
no_dock                   17   0.5%
no_pick_capacity          11   0.3%
no_objective_gain          8   0.2%


### 3.3 Where the moved orders went

In [9]:
div = pd.DataFrame([                                        # one row per moved order
    dict(order=g,
         from_dc=DEF[g]["dc"], to_dc=res2A[g]["chosen_dc"],
         cof_before=DEF[g]["cof"], cof_after=res2A[g]["cof"],
         case_lift=res2A[g]["lift"],                        # extra cases filled
         penalty_saved=DEF[g]["pen"] - res2A[g]["pen"],     # penalty avoided
         extra_freight=res2A[g]["ship"] - DEF[g]["ship"],   # extra shipping paid
         net_gain=objective(res2A[g]) - objective(DEF[g]))  # money gained overall
    for g in FOCUS if res2A[g]["diverted"]])

print("moved:", len(div))
print(div.groupby("to_dc")[["case_lift","penalty_saved","extra_freight","net_gain"]]
         .agg(["count","sum"]).round(0).to_string())

moved: 41
      case_lift         penalty_saved         extra_freight          net_gain          
          count     sum         count     sum         count      sum    count       sum
to_dc                                                                                  
5385          4  1231.0             4   267.0             4   6554.0        4   39814.0
5410          2  1216.0             2    67.0             2   4215.0        2   23007.0
5420          9  2504.0             9  4905.0             9  14173.0        9  130105.0
5490         15  4557.0            15  2518.0            15  27611.0       15  164927.0
5620          5  1227.0             5   712.0             5  17344.0        5   35040.0
5641          3   452.0             3  1022.0             3    948.0        3   26656.0
5773          3   525.0             3     0.0             3   1841.0        3   25061.0


## 4. Sensitivity

One setting changed at a time, everything else held. This shows how much each rule is worth, and
it is the input for the sensitivity discussion in the report.

In [10]:
sens = []
for label, cfg in [
    ("gate as written (5pts + 100 cases)", {}),                      # the doc's rule
    ("no 100-case rule",                   {"MIN_CASE_LIFT": 0.0}),  # let small orders move
    ("stricter gate (10 pts)",             {"MIN_FILL_LIFT_PP": 0.10}),
    ("hold back 50% stock elsewhere",      {"SAFETY_STOCK_FRAC": 0.50}),
]:
    r, _ = greedy(KEYS["2A · by order value"], cfg)   # same sort, one setting changed
    sens.append(metrics(r, label))

S = pd.DataFrame(sens).set_index("scenario")[
        ["orders_diverted","fill_rate","objective_focus","penalty_cost","shipping_cost"]]
S["fill_rate"] = (S["fill_rate"]*100).round(2)
print(S.round(0).to_string())

                                    orders_diverted  fill_rate  objective_focus  penalty_cost  shipping_cost
scenario                                                                                                    
gate as written (5pts + 100 cases)               41       91.0       44810604.0       75259.0       638165.0
no 100-case rule                                 49       91.0       44828565.0       74515.0       645761.0
stricter gate (10 pts)                           29       91.0       44700201.0       77218.0       617496.0
hold back 50% stock elsewhere                    35       91.0       44626803.0       77247.0       628777.0


## 5. Greedy at each MILP sweep size

`05_classical` solves the exact model at several instance sizes and needs the greedy's answer on
**the same subsets** to verify the ceiling at each one.

This cannot be done by filtering the full-472 result down to 50 orders. Greedy run on 50 orders
alone gives a different answer, because it never receives the stock freed by moves among the other
422 — the give-back in the loop above only fires for orders it actually considers. Comparing a
subset-solved MILP against a filtered full-run greedy is how you manufacture a fake negative gap.

So the greedy is genuinely re-run on each subset here.

In [11]:
SWEEP_SIZES = [50, 100, 200, 300, len(FOCUS)]     # must match SWEEP_SIZES in 05_classical

by_size, _all = [], list(FOCUS)
for n in SWEEP_SIZES:
    sub   = sorted(_all, key=lambda g: -HEAD[g]["revenue"])[:n]   # biggest orders first
    FOCUS = sub                                                   # greedy reads this global
    r, _  = greedy(KEYS["2A · by order value"])                   # a real run on this subset
    by_size.append(dict(
        orders    = n,
        baseline  = sum(objective(DEF[g]) for g in sub),          # do nothing, same subset
        objective = sum(objective(r[g])   for g in sub),          # greedy, same subset
        moves     = sum(1 for g in sub if r[g]["diverted"])))
FOCUS = _all                                                      # put the full list back

BY = pd.DataFrame(by_size)
BY["gain_over_baseline"] = BY["objective"] - BY["baseline"]
print(BY.round(0).to_string(index=False))

 orders   baseline  objective  moves  gain_over_baseline
     50 12239830.0 12340615.0      2            100784.0
    100 20281776.0 20432810.0      7            151034.0
    200 32517353.0 32777117.0     12            259764.0
    300 40216520.0 40595549.0     24            379029.0
    472 44365994.0 44810604.0     41            444610.0


## 6. Save

In [12]:
for name, (r, lg) in runs.items():
    tag = "2A" if name.startswith("2A") else "2B"
    to_frame(r, f"greedy_{tag}").to_csv(f"{RESULTS}/results_greedy_{tag}.csv", index=False)

pd.DataFrame(rows).to_csv(f"{RESULTS}/metrics_greedy.csv", index=False)
log2A.to_csv(f"{RESULTS}/greedy_rejection_log.csv", index=False)
BY.to_csv(f"{RESULTS}/metrics_greedy_by_size.csv", index=False)   # read by 05
S.reset_index().to_csv(f"{RESULTS}/greedy_sensitivity.csv", index=False)
print("wrote 6 files")

wrote 6 files


## 7. What this tells us

* **Both sort orders beat Baseline 1**, and they disagree on which orders to move. Value-first
  and shortage-first pick partly different sets. That is normal for a greedy method: whoever goes
  first takes the stock, and nothing is ever redone.
* **The blocker is the business rule, not the warehouse.** In the reject log the C4 gate and
  "this DC does not stock the SKU" dominate. Docks and picks together block a fraction of a
  percent. There is plenty of physical room.
* **The doc's priority rule has nothing to do here.** All 12 priority-8 orders are fully covered
  at their own DC, so none of them reaches the focus list. That rule is already doing its job in
  Stage A, which is why a third priority-based sort would return exactly run 2A.
* **Greedy cannot prove it is right.** It never goes back, so it cannot know whether an early move
  blocked a better one later. That is what `05` is for.